<a href="https://colab.research.google.com/github/hsmu-jeongeun/health-infomatics/blob/main/14_LLM_Nursing_Prompt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 14주차 실습: 간호 인수인계 요약 프롬프트

## 학습 목표
- 대형 언어 모델(LLM)의 작동 원리와 의료 활용 가능성을 이해할 수 있다
- 효과적인 프롬프트 엔지니어링 기법을 적용할 수 있다
- temperature 파라미터가 생성 결과에 미치는 영향을 관찰할 수 있다

---

## 오늘의 핵심 개념: 생성형 AI와 간호 실무

### 보건의료 빅데이터와 LLM
- **14주차 연계:** 보건의료 빅데이터의 최신 활용 트렌드는 생성형 AI(LLM)와의 결합
- **GPT-4 의료 활용 예시:**
  - 진단 요약·처방 리뷰
  - 환자 교육 자료 자동 생성
  - **간호 인수인계 요약** ← 오늘 실습!

### LLM의 작동 원리 (다음 단어 예측)
> 자동완성처럼, LLM은 '다음에 올 가장 그럴듯한 단어'를 반복적으로 예측. 수천억 개의 텍스트로 학습하여 의학 지식도 반영 가능

### 환각(Hallucination)의 위험성 — 의료에서 특히 중요!
- LLM은 그럴듯하지만 **틀린 정보**를 자신감 있게 생성할 수 있음
- 예: 존재하지 않는 약물 용량 제시, 잘못된 처방 코드 생성
- AI 결과를 비판적으로 검토하고, 최종 임상 결정은 반드시 전문가가 확인

### temperature 파라미터
- **0.0에 가까울수록:** 항상 같은 결과 (결정론적, 정확한 정보 필요 시)
- **1.0에 가까울수록:** 창의적이고 다양한 결과 (매번 다름)
- 의료 요약에는 낮은 temperature(0.0~0.3) 권장

In [ ]:
import os
import json
from IPython.display import display, Markdown

print('라이브러리 로드 완료!')
print('이번 실습은 두 가지 방식으로 진행:')
print('  A) OpenAI API 키가 있는 경우 — 실제 GPT 모델 사용')
print('  B) API 키가 없는 경우 — 시뮬레이션 모드로 진행')

라이브러리 로드 완료!
이번 실습은 두 가지 방식으로 진행:
  A) OpenAI API 키가 있는 경우 — 실제 GPT 모델 사용
  B) API 키가 없는 경우 — 시뮬레이션 모드로 진행


## 가상의 환자 경과 기록 (비정형 텍스트)

In [ ]:
# 실제 간호 기록을 모사한 가상의 환자 경과 기록
patient_record = """
[환자 정보]
- 이름: 박OO (보호: 가명)
- 나이: 68세, 여성
- 입원일: 2026-01-10
- 주진단: 제2형 당뇨병(E11) + 고혈압(I10) + 요로감염(N39)

[경과 기록 (최근 24시간)]
오전 06:00 - 체온 38.2°C, 혈압 158/92 mmHg, 맥박 88회/분, SpO2 97%
  → 발열로 해열제(Acetaminophen 500mg) 투약
  → 혈압 상승으로 담당 의사에게 보고, Amlodipine 5mg 추가 처방
  → 소변 혼탁, 악취 있음. 소변 배양검사 시행

오전 09:30 - 공복 혈당 215 mg/dL (목표: <130)
  → 인슐린 Glargine 20단위 피하 투약
  → 식사량 50%로 감소. 저혈당 위험 모니터링 교육
  → 수액 유지(D5W 0.45% NaCl, 80mL/hr)

오후 02:00 - 체온 37.4°C (호전), 혈압 142/88 mmHg
  → 소변 배양 결과: E. coli 양성. 항생제 Cefazolin 1g IV q8h 시작
  → 통증 VAS 4/10, 진통제 요청 없음

오후 06:00 - 혈당 178 mg/dL (호전 중)
  → 식사량 70%로 회복
  → 항생제 투약 완료. 다음 투약 오후 10시 예정
  → 낙상 위험 MORSE 40점 — 침대 난간 올림, 콜벨 교육

오후 11:00 - 활력징후 안정 (BP 138/84, 체온 37.1°C)
  → 수면 취함. 야간 모니터링 지속
"""

print('환자 경과 기록 로드 완료!')
print(f'기록 길이: {len(patient_record)}자')

환자 경과 기록 로드 완료!
기록 길이: 728자


## 프롬프트 엔지니어링 — SBAR 형식 요약

In [ ]:
# =====================
# 수정해 보세요! — 프롬프트를 바꿔보세요
# =====================
prompt_template = """
당신은 숙련된 간호사입니다. 아래의 환자 경과 기록을 읽고 다음 두 가지를 작성하세요:

1. SBAR 형식으로 3줄 이내로 요약 (S-상황, B-배경, A-평가, R-권고)
2. 투약 정보만 따로 목록으로 정리 (약물명, 용량, 경로, 투약 시간)

[환자 경과 기록]
{record}

출력 형식:
## SBAR 요약
...

## 투약 목록
...
"""

prompt = prompt_template.format(record=patient_record)
# =====================

print('프롬프트 준비 완료!')
print(f'프롬프트 길이: {len(prompt)}자')
print('\n--- 프롬프트 미리보기 (첫 300자) ---')
print(prompt[:300] + '...')

프롬프트 준비 완료!
프롬프트 길이: 919자

--- 프롬프트 미리보기 (첫 300자) ---

당신은 숙련된 간호사입니다. 아래의 환자 경과 기록을 읽고 다음 두 가지를 작성하세요:

1. SBAR 형식으로 3줄 이내로 요약 (S-상황, B-배경, A-평가, R-권고)
2. 투약 정보만 따로 목록으로 정리 (약물명, 용량, 경로, 투약 시간)

[환자 경과 기록]

[환자 정보]
- 이름: 박OO (보호: 가명)
- 나이: 68세, 여성
- 입원일: 2026-01-10
- 주진단: 제2형 당뇨병(E11) + 고혈압(I10) + 요로감염(N39)

[경과 기록 (최근 24시간)]
오전 06:00 - 체온 38.2°C, 혈압 ...


## OpenAI API로 실행 (API 키 필요)

> **API 키가 없는 경우:** 다음 셀을 건너뛰고, 시뮬레이션 결과를 확인하세요.

In [ ]:
!pip install openai

  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 4.0 MB/s  0:00:00 eta 0:00:01
Using cached distro-1.9.0-py3-none-any.whl (20 kB)
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
Using cached httpcore-1.0.9-py3-none-any.whl (78 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 16.3 MB/s  0:00:00
Using cached annotated_types-0.7.0-py3-none-any.whl (13 kB)
Using cached h11-0.16.0-py3-none-any.whl (37 kB)
Using cached tqdm-4.67.3-py3-none-any.whl (78 kB)
Using 

In [ ]:
# =====================
# 수정해 보세요!
# =====================
OPENAI_API_KEY = "?"  # OpenAI API 키
temperature    = 0.0  # 0.0 (일관성) ~ 1.0 (창의성)
model_name     = "gpt-4o-mini"        # 모델명
# =====================

USE_OPENAI = OPENAI_API_KEY != "여기에_API_키_입력" and len(OPENAI_API_KEY) > 10

if USE_OPENAI:
    try:
        from openai import OpenAI
        client = OpenAI(api_key=OPENAI_API_KEY)

        response = client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": "당신은 간호 전문가입니다."},
                {"role": "user",   "content": prompt}
            ],
            temperature=temperature,
            max_tokens=800
        )

        api_result = response.choices[0].message.content
        print(f'=== GPT 응답 (temperature={temperature}) ===')
        display(Markdown(api_result))

    except Exception as e:
        print(f'API 오류: {e}')
        USE_OPENAI = False
else:
    print('API 키 미입력 — 아래 시뮬레이션 셀로 이동하세요.')

=== GPT 응답 (temperature=0.0) ===


## SBAR 요약
S: 68세 여성 환자가 제2형 당뇨병, 고혈압, 요로감염으로 입원 중이며, 최근 24시간 동안 발열과 혈압 상승이 관찰됨.  
B: 환자는 체온 38.2°C, 혈압 158/92 mmHg로 시작하여, E. coli에 의한 요로감염 진단 후 항생제 치료를 시작함.  
A: 현재 체온과 혈압이 호전되고 있으며, 혈당도 개선되고 있음.  
R: 지속적인 모니터링과 함께, 다음 항생제 투약을 예정대로 시행하고, 낙상 예방 조치를 강화할 것을 권고함.

## 투약 목록
1. 약물명: Acetaminophen  
   용량: 500mg  
   경로: 경구  
   투약 시간: 오전 06:00  

2. 약물명: Amlodipine  
   용량: 5mg  
   경로: 경구  
   투약 시간: 오전 06:00 (추가 처방)  

3. 약물명: Insulin Glargine  
   용량: 20단위  
   경로: 피하  
   투약 시간: 오전 09:30  

4. 약물명: Cefazolin  
   용량: 1g  
   경로: IV  
   투약 시간: 오후 02:00 (q8h 시작)  

5. 약물명: Cefazolin  
   용량: 1g  
   경로: IV  
   투약 시간: 오후 10:00 (다음 투약 예정)  

## 시뮬레이션 모드 — temperature 효과 비교

> API 키 없이도 temperature 개념을 체험할 수 있는 시뮬레이션입니다.

In [ ]:
# temperature 값에 따른 응답 다양성 시뮬레이션
# (실제 LLM 응답을 모사한 가상 예시)

responses_t0 = [
    """## SBAR 요약 (temperature=0.0 — 동일한 결과)
- S: 68세 여성, 당뇨+고혈압+요로감염으로 입원. 발열 38.2°C, 고혈당 215 mg/dL 관찰
- B: 소변 배양 E. coli 양성 확인. 항생제·인슐린·항고혈압제 투약 중
- R: 혈당 목표치 미달성(목표<130), 항생제 치료 효과 지속 모니터링 필요

## 투약 목록
- Acetaminophen 500mg PO — 06:00 (발열)
- Amlodipine 5mg PO — 추가 처방
- Glargine 20U SC — 09:30 (인슐린)
- Cefazolin 1g IV q8h — 첫 투약 후 매 8시간""",
    """## SBAR 요약 (temperature=0.0 — 동일한 결과)
- S: 68세 여성, 당뇨+고혈압+요로감염으로 입원. 발열 38.2°C, 고혈당 215 mg/dL 관찰
- B: 소변 배양 E. coli 양성 확인. 항생제·인슐린·항고혈압제 투약 중
- R: 혈당 목표치 미달성(목표<130), 항생제 치료 효과 지속 모니터링 필요

## 투약 목록
- Acetaminophen 500mg PO — 06:00 (발열)
- Amlodipine 5mg PO — 추가 처방
- Glargine 20U SC — 09:30 (인슐린)
- Cefazolin 1g IV q8h — 첫 투약 후 매 8시간"""
]

responses_t1 = [
    """## SBAR 요약 (temperature=1.0 — 매번 달라짐)
- S: 고령 당뇨 환자, 열·고혈당·UTI 3중 복합 상태
- B: 요로 감염균(E.coli) 확인으로 항생제 치료 시작. 인슐린·혈압약 병행
- R: 혈당 조절 강화 필요. 감염 치료 반응 및 낙상 위험 주의 요망

## 투약 목록
[해열] Acetaminophen 500mg 경구 06:00
[항고혈압] Amlodipine 5mg 경구 추가
[인슐린] 글라진 20단위 피하 09:30
[항생제] 세파졸린 1g 정맥 8시간 간격""",
    """## SBAR 요약 (temperature=1.0 — 매번 달라짐)
- S: 요로감염 동반 당뇨·고혈압 환자, 활력징후 불안정(발열, 고혈압)
- B: 배양검사에서 E. coli 확인, 항생제 치료 개시. 혈당 215→178 호전 추세
- R: 혈당 목표 130 미달, 항생제 내성 여부 추적관찰, 야간 혈당 측정 지속 필요

## 투약 목록
1) 타이레놀(Acetaminophen) 500mg 오전 6시 복용
2) 혈압약(Amlodipine 5mg) 추가
3) 인슐린 글라진 20IU 피하주사
4) 세파졸린 1g IV — 매 8시간마다 (14:00, 22:00, 06:00)"""
]

print('=== temperature 비교 시뮬레이션 ===')
print()
print('▶ temperature = 0.0 (결과 일관성 높음)')
print('--- 1번째 실행 ---')
print(responses_t0[0])
print('\n--- 2번째 실행 ---')
print(responses_t0[1])
print('\n두 결과가 동일한가?', responses_t0[0] == responses_t0[1])

print()
print('─' * 60)
print()
print('▶ temperature = 1.0 (결과 다양성 높음)')
print('--- 1번째 실행 ---')
print(responses_t1[0])
print('\n--- 2번째 실행 ---')
print(responses_t1[1])
print('\n두 결과가 동일한가?', responses_t1[0] == responses_t1[1])

=== temperature 비교 시뮬레이션 ===

▶ temperature = 0.0 (결과 일관성 높음)
--- 1번째 실행 ---
## SBAR 요약 (temperature=0.0 — 동일한 결과)
- S: 68세 여성, 당뇨+고혈압+요로감염으로 입원. 발열 38.2°C, 고혈당 215 mg/dL 관찰
- B: 소변 배양 E. coli 양성 확인. 항생제·인슐린·항고혈압제 투약 중
- R: 혈당 목표치 미달성(목표<130), 항생제 치료 효과 지속 모니터링 필요

## 투약 목록
- Acetaminophen 500mg PO — 06:00 (발열)
- Amlodipine 5mg PO — 추가 처방
- Glargine 20U SC — 09:30 (인슐린)
- Cefazolin 1g IV q8h — 첫 투약 후 매 8시간

--- 2번째 실행 ---
## SBAR 요약 (temperature=0.0 — 동일한 결과)
- S: 68세 여성, 당뇨+고혈압+요로감염으로 입원. 발열 38.2°C, 고혈당 215 mg/dL 관찰
- B: 소변 배양 E. coli 양성 확인. 항생제·인슐린·항고혈압제 투약 중
- R: 혈당 목표치 미달성(목표<130), 항생제 치료 효과 지속 모니터링 필요

## 투약 목록
- Acetaminophen 500mg PO — 06:00 (발열)
- Amlodipine 5mg PO — 추가 처방
- Glargine 20U SC — 09:30 (인슐린)
- Cefazolin 1g IV q8h — 첫 투약 후 매 8시간

두 결과가 동일한가? True

────────────────────────────────────────────────────────────

▶ temperature = 1.0 (결과 다양성 높음)
--- 1번째 실행 ---
## SBAR 요약 (temperature=1.0 — 매번 달라짐)
- S: 고령 당뇨 환자, 열·고혈당·UTI 3중 복합 상태
- B: 요로 감염균(E.coli) 확인으로 항생제 치료 시작. 인슐린·혈

## LLM 활용 시 간호사의 역할

| AI 역할 | 간호사 역할 |
|---------|------------|
| 긴 기록 → 초안 요약 생성 | 의학적 정확성 검토 |
| 투약 목록 자동 추출 | 용량·경로·시간 재확인 |
| SBAR 구조화 제안 | 임상 맥락 추가 판단 |

> **환각(Hallucination) 주의:** AI가 생성한 약물 정보는 반드시 처방전·약물 데이터베이스와 대조해야 합니다!

---

## Canvas 퀴즈 안내

아래 코드 셀을 실행하여 나온 결과를 Canvas 퀴즈의 정답으로 제출하세요.

**문제:** 실습 코드에서 `temperature` 변수 값을 **0.0에서 1.0으로 올렸을 때**, 요약 결과가 매번 달라지는 현상을 관찰했는가? **(O 또는 X)**

In [ ]:
# [실습] 아래 코드를 실행하여 나온 결과를 Canvas 퀴즈의 정답으로 제출
# temperature=1.0일 때 두 응답이 다르면 -> O
cancer_answer_check = (responses_t1[0] != responses_t1[1])
canvas_answer = 'O' if cancer_answer_check else 'X'
print(f'Answer: {canvas_answer}')